## WeeklyMargin_bySalesCchannels_Report

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
import numpy as np

# === File paths ===
folder_path = r'C:\Users\Victor\Vitos_Work\exe_executive\exe_repositories\exe_margin_metadata_insights'
file_name = 'Weekly_Margin_check.csv'
file_path = os.path.join(folder_path, file_name)

# === PDF class ===
class PDF(FPDF):
    def header(self):
        pass

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

# === Table utility ===
def add_table(pdf, data, column_widths, alignments):
    pdf.set_font("Courier", size=10)
    for row in data:
        for i, cell in enumerate(row):
            pdf.cell(column_widths[i], 10, str(cell), border=1, align=alignments[i])
        pdf.ln()

# === Summary calculator ===
def calculate_summary(df):
    total_quantity = df["Quantity"].sum()
    total_sell = df["Revenue"].sum()
    total_orders = df["Order_No"].nunique()
    total_order_lines = df.shape[0]
    total_profit = df["Profit"].sum()
    avg_order_value = total_sell / total_orders if total_orders else 0
    ips = total_quantity / (total_orders - 1) if total_orders > 1 else 0
    avg_profit_per_order = total_profit / total_orders if total_orders else 0

    return {
        "Total Units Sold": f'{total_quantity:,.1f}',
        "Total Cost ($)": f'${df["Cost"].sum():,.2f}',
        "Total Revenue ($)": f'${total_sell:,.2f}',
        "Total Profit ($)": f'${total_profit:,.2f}',
        "Margin (%)": f'{(total_profit / total_sell) * 100:.2f}%' if total_sell else "0.00%",
        "Average Order Value ($)": f'${avg_order_value:,.2f}',
        "Items Per Sale (IPS)": f'{ips:.2f}',
        "Average Profit per Order ($)": f'${avg_profit_per_order:,.2f}',
        "Number of Orders": f'{total_orders}',
        "Number of Order Lines": f'{total_order_lines}'
    }

# === Save plot helper ===
def save_plot_as_image(fig, filename):
    fig.savefig(filename, bbox_inches='tight', format='png')
    plt.close(fig)

# === Main function ===
def analyze_csv_metadata_and_generate_pdf(file_path):
    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    if not os.path.exists(file_path):
        pdf.set_font("Arial", size=12)
        pdf.cell(0, 10, f"File not found: {file_path}", ln=True)
        pdf.output("Weekly_Margin_Report_Optimized.pdf")
        return

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        pdf.set_font("Arial", size=12)
        pdf.cell(0, 10, f"Error loading file: {e}", ln=True)
        pdf.output("Weekly_Margin_Report_Optimized.pdf")
        return

    df.rename(columns={'Sell': 'Revenue'}, inplace=True)

    def get_sales_channel(order_no):
        if str(order_no).startswith('100'):
            return 'ECOM'
        elif str(order_no).startswith('Shop'):
            return 'BURL'
        elif str(order_no).startswith('Bris'):
            return 'BRIS'
        return 'Other'

    df['Sales_Channel'] = df['Order_No'].apply(get_sales_channel)

    # === Summary & Stats Tables ===
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, "Weekly Margin Tracking and Performance Overview", ln=True)
    pdf.ln(7)

    pdf.cell(0, 10, "Summary Table (Overall)", ln=True)
    pdf.ln(7)
    add_table(pdf, [[k, v] for k, v in calculate_summary(df).items()], [70, 60], ['L', 'C'])
    pdf.ln(7)

    for channel in df['Sales_Channel'].unique():
        pdf.cell(0, 10, f"{channel} Summary Table", ln=True)
        pdf.ln(11)
        add_table(
            pdf,
            [[k, v] for k, v in calculate_summary(df[df['Sales_Channel'] == channel]).items()],
            [70, 60],
            ['L', 'C']
        )
        pdf.ln(11)

    # === Stats Table ===
    pdf.set_font("Arial", 'B', 10)
    pdf.cell(0, 10, "Summary Statistics for Numerical Columns", ln=True)
    pdf.ln(12)

    stats_df = df[['Quantity', 'Cost', 'Revenue', 'Profit', 'Margin']].describe().T.round(2)
    stats_df = stats_df[['mean', '50%', 'min', 'max', 'std']].reset_index()
    stats_df.columns = ['Metric', 'Mean', 'Median', 'Min', 'Max', 'Std Dev']
    stats_df['Metric'] = stats_df['Metric'].replace({'Quantity': 'Units Sold'})
    stats_table = stats_df.values.tolist()
    stats_table.insert(0, stats_df.columns.tolist())
    add_table(pdf, stats_table, [40, 30, 30, 30, 30, 30], ['L', 'C', 'C', 'C', 'C', 'C'])

    # === Metadata Overview ===
    pdf.ln(12)
    pdf.set_font("Arial", 'B', 10)
    pdf.cell(0, 10, "Metadata Overview", ln=True)
    pdf.ln(12)

    meta_counts = {
        'Product_Code': df['Product_Code'].nunique(),
        'Product_Group': df['Product_Group'].nunique(),
        'Subcategory': df['Subcategory'].nunique(),
        'Brand': df['Brand'].nunique(),
        'Units Sold': df['Quantity'].nunique(),
        'Order_Date': df['Order_Date'].nunique(),
        'Cost <= 0': (df['Cost'] <= 0).sum(),
        'Revenue <= 0': (df['Revenue'] <= 0).sum(),
        'Profit <= 0': (df['Profit'] <= 0).sum(),
        'Margin < 0.3': (df['Margin'] < 0.3).sum(),
        'Margin >= 0.3 and < 0.4': ((df['Margin'] >= 0.3) & (df['Margin'] < 0.4)).sum(),
        'Margin >= 0.4 and < 0.5': ((df['Margin'] >= 0.4) & (df['Margin'] < 0.5)).sum(),
        'Margin >= 0.5 and < 0.6': ((df['Margin'] >= 0.5) & (df['Margin'] < 0.6)).sum(),
        'Margin >= 0.6': (df['Margin'] >= 0.6).sum()
    }

    pdf.set_font("Courier", size=10)
    for k, v in meta_counts.items():
        pdf.cell(60, 10, str(k), border=1, align='C')
        pdf.cell(60, 10, str(v), border=1, align='C')
        pdf.ln()

    # === Charts ===
    neg_df = df[df['Margin'] < 0]

    # Helper for clean bar plots (KEEPING YOUR ORIGINAL STYLE)
    def plot_bar_with_labels(data, title, ylabel, filename, palette, hue_values):
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(x=data.index, y=data.values, hue=hue_values, palette=palette, ax=ax, legend=False)
        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis='x', rotation=90)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for p in ax.patches:
            val = f'{int(p.get_height())}' if ylabel == 'Count' else f'${p.get_height():,.0f}'
            ax.annotate(val, (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=8)
        fig.tight_layout()
        save_plot_as_image(fig, filename)

    # ---------------------------
    # Original charts (UNCHANGED)
    # ---------------------------
    plot_bar_with_labels(neg_df['Subcategory'].value_counts().head(15),
                         "N° of Items Sold Under Cost by Subcategory",
                         "Count", "negative_margin_subcategory.png", 'autumn',
                         neg_df['Subcategory'].value_counts().head(15).index)

    plot_bar_with_labels(neg_df['Brand'].value_counts().head(15),
                         "N° of Items Sold Under Cost by Brand",
                         "Count", "negative_margin_brand.png", 'summer',
                         neg_df['Brand'].value_counts().head(15).index)

    plot_bar_with_labels(df.groupby('Subcategory')['Revenue'].sum().nlargest(10),
                         "Top 10 Revenue by Subcategory",
                         "Revenue", "top_10_revenue_subcategory.png", 'viridis',
                         df.groupby('Subcategory')['Revenue'].sum().nlargest(10).index)

    plot_bar_with_labels(df.groupby('Brand')['Revenue'].sum().nlargest(10),
                         "Top 10 Revenue by Brand",
                         "Revenue", "top_10_revenue_brand.png", 'coolwarm',
                         df.groupby('Brand')['Revenue'].sum().nlargest(10).index)

    # === 4 Charts on One Page (UNCHANGED) ===
    pdf.add_page()
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, "Margin & Revenue Bar Chart Overview", ln=True, align='C')

    image_width = 95
    image_height = 100
    top_y = 20
    gap_y = 30
    bottom_y = top_y + image_height + gap_y

    pdf.image("negative_margin_subcategory.png", x=10, y=top_y, w=image_width, h=image_height)
    pdf.image("negative_margin_brand.png", x=105, y=top_y, w=image_width, h=image_height)
    pdf.image("top_10_revenue_subcategory.png", x=10, y=bottom_y, w=image_width, h=image_height)
    pdf.image("top_10_revenue_brand.png", x=105, y=bottom_y, w=image_width, h=image_height)

    # ==========================================================
    # NEW ADDITION (REQUESTED):
    # Top 10 Revenue by Brand + Subcategory for EACH Sales Channel
    # Inserted here: AFTER overall top-10 charts, BEFORE margin histogram
    # ==========================================================
    channel_palettes = {
        "ECOM": "Oranges",
        "BURL": "Blues",
        "BRIS": "Greens"
    }

    # Create 6 charts (2 per channel)
    for ch, pal in channel_palettes.items():
        df_ch = df[df['Sales_Channel'] == ch]
        if df_ch.empty:
            continue

        top_sub = df_ch.groupby('Subcategory')['Revenue'].sum().nlargest(10)
        top_brand = df_ch.groupby('Brand')['Revenue'].sum().nlargest(10)

        plot_bar_with_labels(top_sub,
                             f"Top 10 Revenue by Subcategory - {ch}",
                             "Revenue", f"top_10_revenue_subcategory_{ch}.png", pal,
                             top_sub.index)

        plot_bar_with_labels(top_brand,
                             f"Top 10 Revenue by Brand - {ch}",
                             "Revenue", f"top_10_revenue_brand_{ch}.png", pal,
                             top_brand.index)

    # Add the 6 charts to PDF across 2 pages (4 + 2), keeping your style/layout pattern
    # Page A: BURL + BRIS (4 charts)
    pdf.add_page()
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, "Top 10 Revenue by Sales Channel - Stores", ln=True, align='C')

    image_width = 95
    image_height = 95
    top_y = 20
    gap_y = 20
    bottom_y = top_y + image_height + gap_y

    # BURL row
    if os.path.exists("top_10_revenue_subcategory_BURL.png"):
        pdf.image("top_10_revenue_subcategory_BURL.png", x=10, y=top_y, w=image_width, h=image_height)
    if os.path.exists("top_10_revenue_brand_BURL.png"):
        pdf.image("top_10_revenue_brand_BURL.png", x=105, y=top_y, w=image_width, h=image_height)

    # BRIS row
    if os.path.exists("top_10_revenue_subcategory_BRIS.png"):
        pdf.image("top_10_revenue_subcategory_BRIS.png", x=10, y=bottom_y, w=image_width, h=image_height)
    if os.path.exists("top_10_revenue_brand_BRIS.png"):
        pdf.image("top_10_revenue_brand_BRIS.png", x=105, y=bottom_y, w=image_width, h=image_height)

    # Page B: ECOM (2 charts + histogram on SAME page)
    pdf.add_page()
    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, "Top 10 Revenue by Sales Channel - ECOM", ln=True, align='C')
    
    image_width = 95
    image_height = 80   # slightly smaller to make room
    top_y = 18
    
    if os.path.exists("top_10_revenue_subcategory_ECOM.png"):
        pdf.image("top_10_revenue_subcategory_ECOM.png", x=10, y=top_y, w=image_width, h=image_height)
    if os.path.exists("top_10_revenue_brand_ECOM.png"):
        pdf.image("top_10_revenue_brand_ECOM.png", x=105, y=top_y, w=image_width, h=image_height)
    
    # === Chart 5: Histogram for Margin (with Labels) ===
    bin_count = 30
    bins = np.linspace(df['Margin'].min(), df['Margin'].max(), bin_count + 1)
    fig5 = plt.figure(figsize=(12, 6))
    ax5 = sns.histplot(df['Margin'], bins=bins, kde=True, color='purple', edgecolor='black')
    
    ax5.set_title("Margin Distribution Histogram", fontsize=16, fontweight='bold')
    ax5.set_xlabel("Margin Range")
    ax5.set_ylabel("Frequency")
    
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_labels = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins) - 1)]
    ax5.set_xticks(bin_centers)
    ax5.set_xticklabels(bin_labels, rotation=45, fontsize=8, ha='right')
    
    ax5.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    
    for rect in ax5.patches:
        height = rect.get_height()
        if height > 0:
            ax5.annotate(f'{int(height)}',
                         xy=(rect.get_x() + rect.get_width() / 2, height),
                         xytext=(0, 3),
                         textcoords="offset points",
                         ha='center', va='bottom',
                         fontsize=8)
    
    fig5.tight_layout()
    save_plot_as_image(fig5, "margin_distribution_histogram.png")
    
    # Put histogram right under the ECOM charts (tight spacing)
    pdf.set_font("Arial", 'B', 12)
    pdf.set_xy(10, top_y + image_height + 5)
    pdf.cell(0, 8, "Margin Distribution of all items sold last week", ln=True, align='C')
    pdf.image("margin_distribution_histogram.png", x=10, y=top_y + image_height + 15, w=190)
    

    # === Export PDF ===
    pdf.output("Weekly_Margin_Report_Optimized.pdf")

# === Run ===
if __name__ == '__main__':
    analyze_csv_metadata_and_generate_pdf(file_path)


